# Просмотр датасета OpenAerialMap

Ноутбук для знакомства с корпусом: загрузка манифеста, перемешивание, вывод
случайной пары со всеми её массивами — так же, как в отчёте `PAIR_ANATOMY.html`,
только интерактивно и на любой паре.

**Порядок работы:** выполнить ячейки 1–4 один раз, дальше перезапускать
«**Случайная пара**» столько раз, сколько нужно — каждый запуск берёт следующую
пару из перемешанного пула.

Зависимости — в `requirements.txt` рядом:

```
pip install -r requirements.txt
```

Сопутствующие документы: состав корпуса — `README.md`, как он получен —
`METHODOLOGY.md`, как на нём обучать — `TRAINING.md`, метрики приёмки —
`ACCEPTANCE.md`.

## 1. Загрузка манифеста

`manifest.csv` — единственный источник истины о составе: какие пары есть, в
каком сплите лежат, с каким весом и какой точностью разметки. Файлы `.npz`
читаются **через него**, а не через `glob` по каталогу: так не потеряется
деление на сплиты и не попадут в выборку пары с нулевым весом.

In [ ]:
import csv, json, math, random
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Каталог поставки. Если ноутбук лежит рядом с парами — оставить как есть.
ROOT = Path(".")

rows = list(csv.DictReader((ROOT / "manifest.csv").open(encoding="utf-8")))
print(f"строк в манифесте: {len(rows)}")

missing = [r["pair"] for r in rows if not (ROOT / r["pair"]).exists()][:5]
if missing:
    print("НЕТ ФАЙЛОВ (первые 5):", missing)
else:
    print("все файлы на месте")

def tally(key):
    return dict(sorted(Counter(r[key] for r in rows).items(), key=lambda kv: -kv[1]))

print("\nвид пары:  ", tally("pair_kind"))
print("сплит:     ", tally("split"))
print("класс GT:  ", tally("gt_class"))
print("вес:       ", tally("weight"))
print(f"\nплощадок: {len({r['scene'] for r in rows})}, "
      f"гео-кластеров: {len({r['geo_cluster'] for r in rows})}, "
      f"объём: {sum(int(r['bytes']) for r in rows) / 2**30:.2f} ГБ")

## 2. Перемешивание и отбор

Порядок в манифесте не случаен — пары идут площадками, и первые полсотни строк
дали бы одну и ту же местность. Поэтому пул перемешивается один раз с
фиксированным зерном: просмотр воспроизводим, а соседние пары не родственники.

Фильтры ниже — то, что стоит менять под задачу:

* `SPLIT` — `train` / `val` / `heldout` либо `None` для всех;
* `KIND` — `orto_basemap` (боевой тип), `same_source` (контроль), `cross_date`
  (разные даты съёмки), либо `None`;
* `MIN_WEIGHT` — оставить `0.01`, чтобы не попадали карты рельефа (вес 0).

In [ ]:
SEED = 20260901
SPLIT = None            # "train" | "val" | "heldout" | None
KIND = "orto_basemap"   # "orto_basemap" | "same_source" | "cross_date" | None
MIN_WEIGHT = 0.01

pool = [r for r in rows
        if (SPLIT is None or r["split"] == SPLIT)
        and (KIND is None or r["pair_kind"] == KIND)
        and float(r["weight"]) >= MIN_WEIGHT]
random.Random(SEED).shuffle(pool)
cursor = 0

print(f"в пуле {len(pool)} пар из {len(rows)}")
print("первые пять после перемешивания:")
for r in pool[:5]:
    print(f"  {r['pair'][:46]:46} {r['split']:8} covis {r['covis_frac']:>5} "
          f"h {r['height_m']:>5} м  наклон {r['tilt_deg']:>5}°")

## 3. Чтение пары

Внутри `.npz` шесть массивов. Изображения хранятся **готовыми JPEG в байтах**
(пара весит мегабайты вместо десятков), поле соответствий — `float16` с `NaN`
вне зоны ко-видимости, маска — `uint8`.

Две вещи, о которых легко забыть:

1. `warp` приводится к `float32` — в `float16` арифметика координат теряет
   точность на стороне B шириной до 2048 px;
2. `isfinite` проверяется **даже внутри маски**: маска и конечность значений
   хранятся отдельно, надёжнее опираться на оба признака.

In [ ]:
def load_pair(name):
    """Читает пару и возвращает всё, что в ней есть, уже в рабочих типах."""
    d = np.load(ROOT / name, allow_pickle=False)
    a = cv2.cvtColor(cv2.imdecode(d["image_a_jpeg"], cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    b = cv2.cvtColor(cv2.imdecode(d["image_b_jpeg"], cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    warp = d["warp_ab"].astype(np.float32)          # (H, W, 2) — координаты в B
    mask = d["mask_ab"].astype(bool)                # (H, W)
    finite = np.isfinite(warp).all(axis=-1)
    return dict(name=name, a=a, b=b, warp=warp, mask=mask, finite=finite,
                valid=mask & finite, meta=json.loads(str(d["meta"])))


def correspondences(p):
    """Соответствия как два списка точек — то, что нужно матчеру и метрикам."""
    ys, xs = np.nonzero(p["valid"])
    return np.stack([xs, ys], 1).astype(np.float32), p["warp"][ys, xs]


p = load_pair(pool[0]["pair"])
pa, pb = correspondences(p)
print(f"{p['name']}")
print(f"  A {p['a'].shape}  B {p['b'].shape}  warp {p['warp'].shape} {p['warp'].dtype}")
print(f"  валидных пикселей: {p['valid'].mean():.3f} (covis_frac в meta: {p['meta']['covis_frac']})")
print(f"  соответствий: {len(pa):,}")

## 4. Отрисовка

Функции ниже повторяют то, что показано в `PAIR_ANATOMY.html`: стороны пары,
обе компоненты поля соответствий в псевдоцвете, маска ко-видимости поверх кадра,
регулярная сетка A и её образ в B, точечная трассировка.

Цвет узла сетки кодирует его положение в кадре — поэтому видно не только «куда
легло», но и что взаимный порядок точек сохранился: сетка повернулась и сжалась
целиком, а не рассыпалась.

In [ ]:
def colorize(chan, valid):
    """Компонента поля соответствий в псевдоцвете; вне маски — серое."""
    out = np.zeros(chan.shape + (3,), np.uint8)
    if valid.any():
        lo, hi = np.nanpercentile(chan[valid], [1, 99])
        norm = np.clip((chan - lo) / max(hi - lo, 1e-6), 0, 1)
        norm = np.nan_to_num(norm, nan=0.0)   # NaN при касте в uint8 дают мусор
        out = cv2.cvtColor(cv2.applyColorMap((norm * 255).astype(np.uint8),
                                             cv2.COLORMAP_TURBO), cv2.COLOR_BGR2RGB)
    out[~valid] = (60, 60, 60)
    return out


def draw_grid(p, step=64):
    """Регулярная сетка в A и её образ в B: сама геометрия пары."""
    ga, gb = p["a"].copy(), p["b"].copy()
    h, w = p["valid"].shape
    for y in range(step // 2, h, step):
        for x in range(step // 2, w, step):
            if not p["valid"][y, x]:
                continue
            u, v = p["warp"][y, x]
            col = (int(255 * x / w), int(255 * y / h), 200)
            for img, (px, py) in ((ga, (x, y)), (gb, (int(u), int(v)))):
                if not (0 <= px < img.shape[1] and 0 <= py < img.shape[0]):
                    continue
                cv2.circle(img, (px, py), 5, col, -1, cv2.LINE_AA)
                cv2.circle(img, (px, py), 5, (0, 0, 0), 1, cv2.LINE_AA)
    return ga, gb


def trace_points(p, seed=3):
    """Помеченные точки A и их адреса в B — проверка соответствия глазами."""
    rng = np.random.default_rng(seed)
    ta, tb, table = p["a"].copy(), p["b"].copy(), []
    h, w = p["valid"].shape
    rows_n, cols = 2, 3
    n = 0
    # точки берутся по ячейкам сетки: случайная выборка кучкуется,
    # и половина меток налезает друг на друга
    for gy in range(rows_n):
        for gx in range(cols):
            y0, y1 = gy * h // rows_n, (gy + 1) * h // rows_n
            x0, x1 = gx * w // cols, (gx + 1) * w // cols
            cell = p["valid"][y0:y1, x0:x1]
            if not cell.any():
                continue
            cy, cx = np.nonzero(cell)
            k = int(rng.integers(len(cx)))
            x, y = int(cx[k]) + x0, int(cy[k]) + y0
            u, v = p["warp"][y, x]
            n += 1
            for img, (px, py) in ((ta, (x, y)), (tb, (int(u), int(v)))):
                cv2.drawMarker(img, (px, py), (0, 0, 0), cv2.MARKER_CROSS, 34, 5, cv2.LINE_AA)
                cv2.drawMarker(img, (px, py), (255, 210, 40), cv2.MARKER_CROSS, 30, 2, cv2.LINE_AA)
                cv2.putText(img, str(n), (px + 12, py - 10), cv2.FONT_HERSHEY_SIMPLEX,
                            0.9, (0, 0, 0), 5, cv2.LINE_AA)
                cv2.putText(img, str(n), (px + 12, py - 10), cv2.FONT_HERSHEY_SIMPLEX,
                            0.9, (255, 210, 40), 2, cv2.LINE_AA)
            table.append((n, x, y, float(u), float(v)))
    return ta, tb, table


def side_by_side(left, right, gap=14):
    """Стороны рядом в натуральном размере: кроп B крупнее кадра A."""
    h = max(left.shape[0], right.shape[0])
    out = np.full((h, left.shape[1] + right.shape[1] + gap, 3), 250, np.uint8)
    yl, yr = (h - left.shape[0]) // 2, (h - right.shape[0]) // 2
    out[yl:yl + left.shape[0], :left.shape[1]] = left
    out[yr:yr + right.shape[0], left.shape[1] + gap:] = right
    return out


def show(img, title="", width=13):
    fig, ax = plt.subplots(figsize=(width, width * img.shape[0] / img.shape[1]))
    ax.imshow(img)
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=11, loc="left", color="#444")
    plt.tight_layout()
    plt.show()

## 5. Случайная пара

Ячейка ниже — рабочая: **перезапускать сколько угодно раз**, каждый запуск берёт
следующую пару из перемешанного пула и показывает её целиком.

Чтобы вернуться к конкретной паре, достаточно передать её имя:
`show_pair(load_pair("base_pair_0f2e9280b238_00002_partial.npz"))`.

In [ ]:
MANIFEST_FIELDS = ["split", "pair_kind", "gt_class", "gt_sigma_px", "weight",
                   "geo_cluster", "scene", "modality", "dup_kind", "вердикт"]
META_FIELDS = ["height_m", "tilt_deg", "delta_yaw_deg", "scale_ratio", "gsd_a",
               "gsd_b", "footprint_a_m", "footprint_b_m", "covis_frac",
               "compensation_src", "season_a", "season_b", "date_a", "date_b"]


def show_pair(p, row=None, grid_step=64):
    m = p["meta"]
    print(f"=== {p['name']}")
    if row:
        print("  манифест: " + "  ".join(f"{k}={row[k]}" for k in MANIFEST_FIELDS
                                         if row.get(k) not in (None, "")))
    print("  съёмка:   " + "  ".join(f"{k}={m[k]}" for k in META_FIELDS
                                     if m.get(k) not in (None, "", "None")))
    print(f"  A {p['a'].shape[1]}×{p['a'].shape[0]}   B {p['b'].shape[1]}×{p['b'].shape[0]}"
          f"   валидных {p['valid'].mean():.3f}   NaN в warp {1 - p['finite'].mean():.3f}")

    show(side_by_side(p["a"], p["b"]),
         "стороны пары: слева кадр борта (A), справа кроп подложки (B) — "
         "искать нужно внутри него")
    show(np.concatenate([colorize(p["warp"][..., 0], p["valid"]),
                         np.full((p["a"].shape[0], 14, 3), 250, np.uint8),
                         colorize(p["warp"][..., 1], p["valid"])], axis=1),
         "warp_ab: слева координата u (столбец в B), справа v (строка в B). "
         "Наклон градиента — взаимный поворот сторон; серое — вне маски")

    vis = np.zeros(p["mask"].shape + (3,), np.uint8)
    vis[p["valid"]] = (40, 175, 120)
    vis[~p["valid"]] = (55, 55, 58)
    show(cv2.addWeighted(p["a"], 0.55, vis, 0.45, 0),
         f"mask_ab поверх кадра A: зелёное — соответствие есть "
         f"({100 * p['valid'].mean():.1f} % кадра)", width=8)

    ga, gb = draw_grid(p, grid_step)
    show(side_by_side(ga, gb),
         "регулярная сетка A и её образ в B: цвет узла кодирует положение в кадре")
    ta, tb, table = trace_points(p)
    show(side_by_side(ta, tb),
         "точечная трассировка: метка N слева и метка N справа — один и тот же "
         "кусок земли")
    print("  №   пиксель A (x, y)   →   пиксель B (u, v)")
    for i, x, y, u, v in table:
        print(f"  {i}   ({x:4}, {y:4})      →   ({u:8.1f}, {v:8.1f})")


def next_pair():
    """Следующая пара из перемешанного пула."""
    global cursor
    row = pool[cursor % len(pool)]
    cursor += 1
    return load_pair(row["pair"]), row


p, row = next_pair()
show_pair(p, row)

## 6. Числовые проверки пары

Взглядом видно, что соответствие легло правильно; здесь то же самое числами.
Проверяется не модель, а **сам файл**: согласованность маски с метаданными,
попадание координат внутрь стороны B и гладкость поля.

Поле соответствий построено проективно, поэтому его вторая разность мала:
скачки означали бы порванную разметку. Сравнивать её надо не с нулём, а с
**шагом квантования `float16`** — на координатах порядка 1500 px он и сам около
1 px, и ниже него поле «гладким» быть не может по способу хранения.

In [ ]:
def check_pair(p):
    m, w, v = p["meta"], p["warp"], p["valid"]
    hb, wb = p["b"].shape[:2]
    uv = w[v]
    inside = ((uv[:, 0] >= 0) & (uv[:, 0] < wb) &
              (uv[:, 1] >= 0) & (uv[:, 1] < hb)).mean()
    # вторая разность поля: мера гладкости, у проективного переноса ≈ 0
    d2 = np.abs(np.diff(np.where(v[..., None], w, np.nan), n=2, axis=1))
    smooth = np.nanpercentile(d2, 99) if np.isfinite(d2).any() else float("nan")
    print(f"  ко-видимость: маска {v.mean():.4f} против covis_frac {m['covis_frac']} в meta")
    print(f"  координаты внутри стороны B: {100 * inside:.2f} %")
    print(f"  разброс u: {uv[:, 0].min():8.1f} … {uv[:, 0].max():8.1f}  (B шириной {wb})")
    print(f"  разброс v: {uv[:, 1].min():8.1f} … {uv[:, 1].max():8.1f}  (B высотой {hb})")
    step = float(np.spacing(np.float16(max(uv.max(), 1))))   # шаг хранения float16
    print(f"  гладкость поля (99-й перцентиль второй разности): {smooth:.4f} px "
          f"при шаге квантования float16 {step:.4f} px")
    print(f"  ожидаемая ошибка разметки этой пары: {row['gt_sigma_px']} px "
          f"(класс {row['gt_class']})")

check_pair(p)

## 7. Ресайз: где чаще всего ломается обучение

Тренеры работают с квадратным входом, а стороны пары разного размера — кадр
1024×576, кроп B от 768 до 2048 px. При ресайзе координаты `warp` **обязаны**
масштабироваться вместе с изображением, причём по осям по-разному, а маска
интерполируется только `INTER_NEAREST`: билинейная размывает границу
ко-видимости и добавляет пикселей, у которых соответствия нет.

Ячейка делает ресайз ровно так, как в загрузчике из `TRAINING.md`, и показывает
результат — если координаты пересчитаны неверно, метки разъедутся сразу же.

In [ ]:
def resized_pair(p, size=560):
    """Приведение к квадрату size×size с пересчётом координат warp."""
    hb, wb = p["b"].shape[:2]
    warp = cv2.resize(p["warp"], (size, size), interpolation=cv2.INTER_NEAREST)
    warp[..., 0] *= size / wb          # масштаб по осям разный —
    warp[..., 1] *= size / hb          # забыть об этом и есть частая ошибка
    mask = cv2.resize(p["valid"].astype(np.uint8), (size, size),
                      interpolation=cv2.INTER_NEAREST).astype(bool)
    return dict(name=p["name"] + f" [{size}×{size}]",
                a=cv2.resize(p["a"], (size, size), interpolation=cv2.INTER_AREA),
                b=cv2.resize(p["b"], (size, size), interpolation=cv2.INTER_AREA),
                warp=warp, mask=mask, finite=np.isfinite(warp).all(-1),
                valid=mask & np.isfinite(warp).all(-1), meta=p["meta"])


r = resized_pair(p)
ga, gb = draw_grid(r, 48)
show(side_by_side(ga, gb),
     "после ресайза к 560×560 с пересчётом координат: сетка обязана лежать так же")

## 8. Корпус целиком

Распределения по всему пулу — чтобы видеть, из чего он состоит, а не только
отдельные пары. Высоты 175–300 м, наклоны до 10° (у контрольной оси до 20°),
ко-видимость от 0.5 у частичных компоновок до 1.0 у полных.

In [ ]:
fields = [("height_m", "высота съёмки, м"), ("tilt_deg", "наклон от надира, °"),
          ("covis_frac", "ко-видимость"), ("scale_ratio", "отношение масштабов gsd_a/gsd_b")]
fig, axes = plt.subplots(2, 2, figsize=(12, 6.5))
for ax, (key, title) in zip(axes.ravel(), fields):
    vals = np.array([float(r[key]) for r in pool if r[key] not in ("", None)])
    ax.hist(vals, bins=40, color="#4a7ba7", edgecolor="white", linewidth=0.4)
    ax.set_title(f"{title}   медиана {np.median(vals):.2f}", fontsize=10.5, loc="left")
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print("состав пула по площадкам: "
      f"{len({r['scene'] for r in pool})} площадок, "
      f"крупнейшая даёт {Counter(r['scene'] for r in pool).most_common(1)[0][1]} пар")

---

**Дальше:** обучающий загрузчик, подмена GT в тренерах, сэмплер по весам и
метрики приёмки — в `TRAINING.md`. Числа базовой линии пяти матчеров на этом
корпусе — в `ACCEPTANCE.md` и `MATCHERS_REPORT.html`.